# C3NL Jupyter notebook examples
## How to run commands on your study data
by Greg Scott (gregory.scott99@imperial.ac.uk)

This example shows you how to:
* Arrange your study data directory structure 
* Step over your data structure
* Run bash commands on your data in serial or parallel 

### Preamble
To use this example, first you will need to:
1. Download an unzip the C3NL example data as described below *or*
2. Make your own directory structure (and get some files) which follows the example below
3. Upload the `hpcrunarrayjob.sh` script to the same directory as this notebook

#### If you want to use the C3NL example data

You can download a zip file of example MRI data for 3 traumatic brain injury (TBI) and 3 control participants, covering a number of imaging sequences. Note that the file is ~1.6Gb in size so may take some time to download even on to the HPC.

##### Download the example data file and transfer it to the HPC
The file is available from the Imperial Box site: https://imperialcollegelondon.box.com/shared/static/6prlwdhq12fsahy6hai4gt4h1ksdpqtm.zip

To download this directly into your work directory on the HPC (e.g. `/work/username`), where it is recommended large data files are stored, you can use the `wget` command. This will retrieve the file straight from Box as a direct download. 

This could be done in a notebook cell but (because it can take several minutes) is better done through a terminal. You can open a terminal using SSH (to `login.cx1.hpc.imperial.ac.uk`) or much easier is to open one through Jupyter, by clicking on File -> New -> Terminal).

```bash
wget https://imperialcollegelondon.box.com/shared/static/6prlwdhq12fsahy6hai4gt4h1ksdpqtm.zip -O $WORK/c3nl-example-data-raw.zip
```

The destination here is your work directory, using the environment variable `$WORK` (which should be set for you by the HPC). The `-O` option names the destination file.

##### Unzip the example data file
Unzip the example data file in the terminal.

This will print a list of the files being unzipped. You can add the `-qq` option here (quiet) to avoid lots of output about the files unzipped. Note again that this can take a very long time to complete, and you should wait until it has finished before continuing below.

```bash
unzip -qq $WORK/c3nl-example-data-raw.zip 
```

#### If you want to create your own study data directory structure
If you are creating a new directory structure with your own data, there are many ways to do this. On the HPC, you can use the `mkdir` command to make directories (where the `-p` option is helpful as it will make parent directories, where they do not exist). 

Whichever method you use, we recommend you create a directory structure which follows the subject, visit, modality structure used below.

### Define the location of your data directory structure
It is good practice to define in a variable the root location (or path) of the directory containing your data. That way, if you need to move your whole project to another location, you need only change your code in one place, i.e. changing the value of that variable, rather than have to search for and update wherever you've used the path in your scripts.

Here we use the convention that variables for this kind of purpose are defined in `CAPITALS` (as distinguished from normal variables)

We set our location to be that of the unzipped example data (in your `$WORK` directory (i.e. `/work/username`). If you're using your own data structure, you will need to set `DATADIR` to point to that.

*(Note - to run a cell in the Jupyter notebook, select the cell then press the Play icon on the toolbar or press Ctrl-Enter)*

*(Note - bash commands (which make up the interactive parts of this Matlab notebook) in Jupyter must begin with an additional `!` character (if a single line) or using the notation `%%shell`, if a multi-line cell). Without this, the notebook will interpret the code as Matlab (or whatever language the notebook is set up for).* 

In [46]:
!DATADIR=$WORK/sampledata/c3nl-example-data-raw
pwd

### Looking at your directory structure
We can now look at the data structure.  There are many ways to do this. Here we shall use the `find` command to search for and list all the files and directories contained within our root (`${DATADIR}`) directory, limited to items matching certain criteria.

Here for showing the downloaded example data we use the `-type d` (for showing directories only) option.

The bottom-level directories are `DICOM` directories. These contain the DICOM files from the scanner. These directories in turn are contained within higher-level directories for each subject, visit and modality. 

In [4]:
!find ${DATADIR} -type d

/work/ngraham/sampledata/c3nl-example-data-raw/
/work/ngraham/sampledata/c3nl-example-data-raw/raw
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/crt
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/crt/DICOM
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/dti
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/dti/DICOM
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/rest
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/rest/DICOM
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/t1
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/t1/DICOM
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v2
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v2/crt
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v2/crt/DICOM
/work/ngraham/sampledata/c3n

This view (which only shows only the directories) illustrates the structure we recommend, i.e. having directories structured by:

```root/top-level directory/subject/visit/modality```

#### Listing all the files
To see all the files (as well as directories) you've downloaded into the root data directory in one would take a lot of space in this notebook, so we'd recommend looking using a terminal for this. For example, can run the find command in a terminal with no options, e.g.:

```bash
DATADIR=$WORK/c3nl-example-data-raw
find ${DATADIR}```

or use the `ls` (list) and `cd` (change directory) options, e.g.:

```bash
DATADIR=$WORK/c3nl-example-data-raw
cd ${DATADIR}/raw
ls```

## Stepping over the data directory structure

To execute the same command on every item of data in your structure, you could do this manually, running the command separately for each input file. But a better way is to automatically step over your data structure and run the same command on each file.

There are many ways to do this. You could ignore the directory folder structure completely and get a list of files to run your command on, e.g. from the output of the `find` command.

Instead, **we recommend stepping over the data directory structure in a logical way which maps on to the levels of directories representing subjects, visits and modalities**. By doing this, you can have a handle (actually, a variable) on these each of these aspects (the current subject, visit, modality) as you go. You can use the value of the variables to do useful things.

### Creating new subdirectories in your directory structure
Here, for example, we use walk over our directory structure with nested `for` loops that follow our directory structure. In this case we do this to make a new directory for each subject, visit and modality, which we call `nifti`. This directory will contain the NIFTI version of our raw DICOM files, alongside the `DICOM` directory. 

At each iteration of the inner most `for` loop we make the directory, after first printing the current subject, visit and modality we are operating on. Look at the order of the output and compare this to the structure of the `for` loops.

In [43]:
%%shell
    for subject in `ls ${DATADIR}/raw`
    do
        for visit in `ls ${DATADIR}/raw/${subject}`
        do
            for modality in `ls ${DATADIR}/raw/${subject}/${visit}`
            do
                echo ${subject}/${visit}/${modality}

                mkdir -p ${DATADIR}/raw/${subject}/${visit}/${modality}/nifti
            done;
        done
    done;

con_001/v1/crt
con_001/v1/dti
con_001/v1/rest
con_001/v1/t1
con_001/v2/crt
con_001/v2/dti
con_001/v2/rest
con_001/v2/t1
con_002/v1/crt
con_002/v1/dti
con_002/v1/rest
con_002/v1/t1
con_002/v2/crt
con_002/v2/dti
con_002/v2/rest
con_002/v2/t1
con_003/v1/crt
con_003/v1/dti
con_003/v1/rest
con_003/v1/t1
con_003/v2/crt
con_003/v2/dti
con_003/v2/rest
con_003/v2/t1
pat_001/v1/crt
pat_001/v1/dti
pat_001/v1/rest
pat_001/v1/t1
pat_001/v2/crt
pat_001/v2/dti
pat_001/v2/rest
pat_001/v2/t1
pat_002/v1/crt
pat_002/v1/dti
pat_002/v1/rest
pat_002/v1/t1
pat_002/v2/crt
pat_002/v2/dti
pat_002/v2/rest
pat_002/v2/t1
pat_003/v1/crt
pat_003/v1/dti
pat_003/v1/rest
pat_003/v1/t1
pat_003/v2/crt
pat_003/v2/dti
pat_003/v2/rest
pat_003/v2/t1


Each `for` loop works by using the `ls` command to list the contents of a directory. Each iteration of the `for` loop works on each item from the directory listing. By nesting the `for` loops and moving down through the directory structure, we can iterate over each modality, for each visit, for each subject.


## Running commands on your data in serial

As well as creating new directories for each of your data files, we can run image processing commands. 

Here we use the `dcm2niix` command to convert each DICOM into NIFTI format. We use the previous step (where we created NIFTI directories for each data item) so that the conversion command has an output destination for each file.

*(Note the `dcm2niix` command is part of mricrogl which, at the time of writing, can be made available using the `module load` command with the wrongly-spelt module `mricogl`)*

*(Note - actually executing this will take a long time! You can interrupt the kernel by pressing Escape then tapping the I key twice, or by pressing the Stop button on the tool bar. Sometimes, interrupting the kernel causes problems and you will need to restart it by clicking on Kernel -> Restart Kernel.*

In [ ]:
%%shell
    module load mricogl
    for subject in `ls ${DATADIR}/raw`
    do
        for visit in `ls ${DATADIR}/raw/${subject}`
        do
            for modality in `ls ${DATADIR}/raw/${subject}/${visit}`
            do
                dcm2niix -o ${DATADIR}/raw/${subject}/${visit}/${modality}/nifti -z y ${DATADIR}/raw/${subject}/${visit}/${modality}/DICOM
            done;
        done
    done;

Chris Rorden's dcm2niiX version v1.0.20170724 GCC4.4.7 (64-bit Linux)


## Running commands in parallel as standard PBS jobs (1-50 jobs)

One of the main benefits of using a computing cluster is parallelisation. This lets us perform multiple tasks (or jobs) simultaneously, potentially dramatically reducing the time to carry out analyses compared to using a personal computer.

Here we run exactly the same commands as above but in parallel, using the `qsub` command, which is part of a job management system called PBS. You can read more about submitting jobs at http://www.imperial.ac.uk/admin-services/ict/self-service/research-support/rcs/support/getting-started/

Notice how the `qsub` command used below ends with the very same `dcm2niix` command as above, but additional information about the resources the job needs must be specified. This includes the number of compute nodes (`select=1`) and cores on each node (`ncpus`), the amount of memory it needs (`mem`) and, importantly, the *walltime*. The walltime is your estimate (in `hh:mm:ss`) of the upper limit of the amount of time your job will take to execute. Importantly, **if the walltime of your job is exceeded, the job will be killed before it is finished**. 

*Note, an important limitation of the Imperial HPC, when submitting standard jobs, is that currently a maximum of 50 jobs can be submitted and queued per user at any time. If more than 50 need to be submitted at once, then **array jobs** should be used, which are slightly different, and described next. Here we limit the modality to only the T1, to limit the total number of jobs submitted in one go to less than 50, for this example.*

Run the cell to submit jobs to the cluster for every subject and visit, to convert the T1 DICOM to NIFTI.

In [ ]:
%%shell
    module load mricogl
    for subject in `ls ${DATADIR}/raw`
    do
        for visit in `ls ${DATADIR}/raw/${subject}`
        do
            modality="t1";

            qsub -lselect=1:ncpus=1:mem=8Gb -l walltime=01:00:00 -- module load mricogl; dcm2niix -o ${DATADIR}/raw/${subject}/${visit}/${modality}/nifti -z y ${DATADIR}/raw/${subject}/${visit}/${modality}/DICOM
        done
    done;

1640350.cx1
Chris Rorden's dcm2niiX version v1.0.20170724 GCC4.4.7 (64-bit Linux)
Found 160 DICOM image(s)
Convert 160 DICOM as /work/gps99/c3nl-example-data-raw/raw/con_001/v1/t1/nifti/DICOM_MPRAGE_ADNI_P2_20140220100058_2a (240x256x160x1)
compress: "/usr/bin/pigz" -n -f "/work/gps99/c3nl-example-data-raw/raw/con_001/v1/t1/nifti/DICOM_MPRAGE_ADNI_P2_20140220100058_2a.nii"
Conversion required 0.944992 seconds (0.250000 for core code).
1640351.cx1
Chris Rorden's dcm2niiX version v1.0.20170724 GCC4.4.7 (64-bit Linux)
Found 160 DICOM image(s)
Convert 160 DICOM as /work/gps99/c3nl-example-data-raw/raw/con_001/v2/t1/nifti/DICOM_MPRAGE_ADNI_P2_20150226141630_2a (240x256x160x1)
compress: "/usr/bin/pigz" -n -f "/work/gps99/c3nl-example-data-raw/raw/con_001/v2/t1/nifti/DICOM_MPRAGE_ADNI_P2_20150226141630_2a.nii"
Conversion required 1.297195 seconds (0.260000 for core code).
1640352.cx1
Chris Rorden's dcm2niiX version v1.0.20170724 GCC4.4.7 (64-bit Linux)
Found 160 DICOM image(s)
Convert 160 DIC

Executing this cell should produce a list of job identifiers (ending in `.cx1`).  This is a unique identifier for your job, which you can use to query PBS about the status of your job, and potentially kill the job if you so wish.

You can check on the status of your jobs using the `qstat` command. See http://www.imperial.ac.uk/admin-services/ict/self-service/research-support/rcs/support/getting-started/ for more information.

In [50]:
!qstat

Job id            Name             User              Time Use S Queue
----------------  ---------------- ----------------  -------- - -----
1641887[].cx1     tmp.JWgBKOexbP   ngraham                  0 B v1_throughput24 


To kill a job, use the `qdel` command, followed by your job identifier. To kill all your jobs, you can use:
`qselect | xargs qdel`

#### Knowing what happened to your jobs

All PBS jobs can produce text output and error outputs into two files (or 'streams'), one file ending in `.oXXXXXXX` and another ending in `.eXXXXXXX`. A pair of these files is produced for each of your jobs that has actually executed. These by default appear in the current directory when the jobs were executed. You can load them from the Jupyter 'Files' tab (if there are any to see), or use a command like `cat` to view their contents on the terminal.

#### Viewing the contents of your directory structure

Once your processing has finished, you can view the contents of the same directory structure. Here we limit this to only files that appear in the new NIFTI directories. If our conversion jobs have worked we should expect to see some files there.

In [48]:
!find ${DATADIR} -wholename "*.nii" -type f

/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/crt/nifti/con_001_v1_crt.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/dti/nifti/con_001_v1_dti.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/rest/nifti/con_001_v1_rest.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/t1/nifti/con_001_v1_t1.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v2/crt/nifti/con_001_v2_crt.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v2/dti/nifti/con_001_v2_dti.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v2/rest/nifti/con_001_v2_rest.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v2/t1/nifti/con_001_v2_t1.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_002/v1/crt/nifti/con_002_v1_crt.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_002/v1/dti/nifti/con_002_v1_dti.nii
/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_002/v1/rest/nifti/con_002_v1_rest.n

## Running commands in parallel as PBS array jobs (1+ jobs)
An alternative to standard jobs is to use array jobs. Array jobs are treated on the HPC as a group of related jobs. There is not the stringent limit on the number of subjobs within an array job, i.e. you can have >100s.

An array job works by executing N similar subjobs separately, passing each subjob an integer 1..N. Here we use the looping approach as above, but in doing so, write a text file (here called `commands.txt`) containing the separate N commands to run on your data. Then, this text file (with one command per line) is passed to a simple utility script which we provide called `hpcrunarrayjob.sh`. This script takes this command file and runs each line as a subjob of one overall array job. The walltime, memory and number of cores are provided as above. 

Read more about array jobs at https://www.imperial.ac.uk/admin-services/ict/self-service/research-support/rcs/computing/high-throughput-computing/array-jobs/

In [21]:
!pwd

/export111/work/ngraham/notebooks


In [40]:
%%shell
echo -n "" > commands.txt
for subject in `ls ${DATADIR}/raw`
do
    for visit in `ls ${DATADIR}/raw/${subject}`
    do
        for modality in `ls ${DATADIR}/raw/${subject}/${visit}`
        do
            rm -f ${DATADIR}/raw/${subject}/${visit}/${modality}/nifti/*;
            echo "module load \"mricogl\"; dcm2niix -o ${DATADIR}/raw/${subject}/${visit}/${modality}/nifti -z n ${DATADIR}/raw/${subject}/${visit}/${modality}/DICOM" >> commands.txt
        done;
    done
done;
chmod 755 ./hpcrunarrayjob.sh 
./hpcrunarrayjob.sh commands.txt 01:00:00 1 8Gb

Walltime = 01:00:00
Number of CPUs = 1
Memory = 8Gb
PBS file = 
/tmp/tmp.GxpKfAbpf1
PBS job id = 
1643223[].cx1


In [32]:
%%shell
    PBS_ARRAY_INDEX=2
    cmd=`sed "${PBS_ARRAY_INDEX}q;d" /home/ngraham/9139.14002.txt`
    echo ${cmd}

module load "mricogl"; dcm2niix -o /work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/dti/nifti -z n /work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/dti/DICOM


In [34]:
!cat /tmp/tmp.9lxm7yjgYm

#!/bin/sh
#PBS -l walltime=01:00:00
#PBS -l select=1:ncpus=1:mem=8Gb
#PBS -J 1-48
cmd=`sed "${PBS_ARRAY_INDEX}q;d" /home/ngraham/9139.14002.txt`
${cmd}


In [42]:
!qstat

Job id            Name             User              Time Use S Queue
----------------  ---------------- ----------------  -------- - -----
1643223[].cx1     tmp.GxpKfAbpf1   ngraham                  0 Q v1_throughput24 


In [38]:
!rm *.o*
!rm *.e*

rm: cannot remove ‘*.e*’: No such file or directory


In [15]:
!qselect | xargs qdel

# Some renaming thing

In [44]:
%%shell
    for subject in `ls ${DATADIR}/raw`
    do
        for visit in `ls ${DATADIR}/raw/${subject}`
        do
            for modality in `ls ${DATADIR}/raw/${subject}/${visit}`
            do
                echo ${subject}/${visit}/${modality}
                mv ${DATADIR}/raw/${subject}/${visit}/${modality}/nifti/*.nii ${DATADIR}/raw/${subject}/${visit}/${modality}/nifti/${subject}_${visit}_${modality}.nii;
            done;
            
            mv ${DATADIR}/raw/${subject}/${visit}/dti/nifti/*.bvec ${DATADIR}/raw/${subject}/${visit}/${modality}/nifti/${subject}_${visit}.bvec;
            mv ${DATADIR}/raw/${subject}/${visit}/dti/nifti/*.bval ${DATADIR}/raw/${subject}/${visit}/${modality}/nifti/${subject}_${visit}.bval;
        done
    done;

con_001/v1/crt
con_001/v1/dti
con_001/v1/rest
con_001/v1/t1
con_001/v2/crt
con_001/v2/dti
con_001/v2/rest
con_001/v2/t1
con_002/v1/crt
con_002/v1/dti
con_002/v1/rest
con_002/v1/t1
con_002/v2/crt
con_002/v2/dti
con_002/v2/rest
con_002/v2/t1
con_003/v1/crt
con_003/v1/dti
con_003/v1/rest
con_003/v1/t1
con_003/v2/crt
con_003/v2/dti
con_003/v2/rest
con_003/v2/t1
pat_001/v1/crt
pat_001/v1/dti
pat_001/v1/rest
pat_001/v1/t1
pat_001/v2/crt
pat_001/v2/dti
pat_001/v2/rest
pat_001/v2/t1
pat_002/v1/crt
pat_002/v1/dti
pat_002/v1/rest
pat_002/v1/t1
pat_002/v2/crt
pat_002/v2/dti
pat_002/v2/rest
pat_002/v2/t1
pat_003/v1/crt
pat_003/v1/dti
pat_003/v1/rest
pat_003/v1/t1
pat_003/v2/crt
pat_003/v2/dti
pat_003/v2/rest
pat_003/v2/t1


Check on the status of your array job. You should see only one job listed, with a `[]` square bracket notation indicating this is an array job.

In [25]:
!cat /tmp/tmp.i1JW3jJykH

#!/bin/sh
#PBS -l walltime=01:00:00
#PBS -l select=1:ncpus=1:mem=4Gb
#PBS -J 1-96
cmd=`sed "${PBS_ARRAY_INDEX}q;d" /home/ngraham/8130.11454.txt`
${cmd}


In [21]:
!which dcm2niix

/apps/mricogl/build/mricrogl_lx/dcm2niix


In [7]:
!cat /tmp/tmp.KAYSOJUxkY

#!/bin/sh
#PBS -l walltime=01:00:00
#PBS -l select=1:ncpus=1:mem=8Gb
#PBS -J 1-48
cmd=`sed "${PBS_ARRAY_INDEX}q;d" /home/ngraham/23813.1592.txt`
${cmd}


In [28]:
%%shell
    PBS_ARRAY_INDEX=1
    cmd=`sed "${PBS_ARRAY_INDEX}q;d" /home/ngraham/23813.1592.txt`
    echo ${cmd}

/apps/mricogl/build/mricrogl_lx/dcm2niix -o /work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/crt/nifti -z y /work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/crt/DICOM


In [30]:
!/apps/mricogl/build/mricrogl_lx/dcm2niix -o /work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/crt/nifti -z y /work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/crt/DICOM

Chris Rorden's dcm2niiX version v1.0.20170724 GCC4.4.7 (64-bit Linux)
Found 126 DICOM image(s)
slices stacked despite varying acquisition numbers (if this is not desired please recompile)
Convert 126 DICOM as /work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/crt/nifti/DICOM_fMRI_task_CRT0_20140220100058_11 (64x64x35x126)
compress: "/usr/bin/pigz" -n -f "/work/ngraham/sampledata/c3nl-example-data-raw/raw/con_001/v1/crt/nifti/DICOM_fMRI_task_CRT0_20140220100058_11.nii"
Conversion required 0.873553 seconds (0.260000 for core code).


In [10]:
!qselect | xargs qdel

In [36]:
!pwd

/export111/work/ngraham/notebooks


In [39]:
!rm *.o*;
!rm *.e*;

In [51]:
!qstat

To get more information on any of your jobs, use the `-f` option

In [17]:
!qstat -f

Job Id: 1639390[].cx1
    Job_Name = tp731f0967_d132_4ae7_b072_d971f47119c7
    Job_Owner = gps99@login-2-internal.cx1.hpc.ic.ac.uk
    job_state = B
    queue = v1_throughput24
    server = cx1
    Checkpoint = u
    ctime = Wed Apr 25 23:16:38 2018
    Error_Path = login-2-internal.cx1.hpc.ic.ac.uk:/export131/home/gps99/PTAPIP
	ELINE_CODE/eeg/hpc/tp731f0967_d132_4ae7_b072_d971f47119c7.e1639390.^arr
	ay_index^
    Hold_Types = n
    Join_Path = n
    Keep_Files = n
    Mail_Points = a
    mtime = Thu Apr 26 08:27:15 2018
    Output_Path = login-2-internal.cx1.hpc.ic.ac.uk:/export131/home/gps99/PTAPI
	PELINE_CODE/eeg/hpc/tp731f0967_d132_4ae7_b072_d971f47119c7.o1639390.^ar
	ray_index^
    Priority = 0
    qtime = Wed Apr 25 23:16:38 2018
    Rerunable = True
    Resource_List.mem = 16gb
    Resource_List.mpiprocs = 2
    Resource_List.ncpus = 2
    Resource_List.nodect = 1
    Resource_List.place = free
    Resource_List.select = 1:ncpus=2:mem=16gb:mpiprocs=2:ompthreads=1
    Resource_L

## What next?

You can use the same looping approach used here within the HPC to parallelise other jobs. For example, you could write code to reorient the converted NIFTI T1s. You could write code to rename the NIFTI T1 files (whereas the raw filename, converted from the DICOM, is subject-dependent) into standard names like `t1.nii.gz`. You could write code to create a `preprocessed` directory, alongside the `raw` top level directory, in which to store these preprocessed files. 